# SENSEI — Session Intelligence
## Module 2 · Purchase Prediction — Classification

Binary classification on the session feature store built in Module 1.
Target: `purchased` (1 = session contains a transaction). Purchase rate: 0.81 % — imbalance ratio ~1:122.

## Contents
1. Load feature store & train/test split
2. Logistic Regression (baseline)
3. Random Forest
4. Evaluation: normalized confusion matrix + PR curve
5. Cross-validation (5-fold stratified)
6. Permutation feature importance
7. Threshold tuning

## Summary

| | |
|---|---|
| Models | Logistic Regression (baseline), Random Forest |
| Imbalance handling | `class_weight='balanced'` |
| Primary metrics | PR-AUC, F1 — not accuracy |
| CV | 5-fold stratified |
| Feature importance | Permutation (unbiased vs. MDI) |
| Threshold | Tuned to maximise F1 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    f1_score,
    ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = os.path.join('..', 'data')
RANDOM_STATE = 42

## 1. Load Data & Train/Test Split

In [ ]:
sessions = pd.read_parquet(os.path.join(DATA_DIR, 'sessions_features.parquet'))
print(f'Shape: {sessions.shape}')
sessions.head()

In [ ]:
FEATURES = [
    'n_views',
    'n_addtocart',
    'n_items',
    'n_revisited_items',
    'duration_sec',
    'hour_of_day',
    'day_of_week',
    'view_to_cart_ratio',
    'is_first_session',
]
TARGET = 'purchased'

X = sessions[FEATURES]
y = sessions[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Purchase rate — train: {y_train.mean()*100:.2f}%  |  test: {y_test.mean()*100:.2f}%')

In [ ]:
# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 2. Baseline: Logistic Regression

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)

y_pred_lr   = lr.predict(X_test_scaled)
y_proba_lr  = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['no purchase', 'purchase']))

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['no purchase', 'purchase']))

## 4. Evaluation & Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Normalized confusion matrices ---
for ax, y_pred, label in [
    (axes[0], y_pred_lr, 'Logistic Regression'),
    (axes[1], y_pred_rf, 'Random Forest'),
]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['No purchase', 'Purchase'],
        normalize='true',          # show recall per class, not raw counts
        ax=ax, colorbar=False, cmap='Blues',
        values_format='.2f',
    )
    ax.set_title(f'{label}\n(normalized by true label)')

# --- Precision-Recall curves ---
for y_prob, label, color in [
    (y_proba_lr, 'Logistic Regression', '#4C72B0'),
    (y_proba_rf, 'Random Forest',        '#55A868'),
]:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[2].plot(rec, prec, label=f'{label} (AP={ap:.3f})', color=color)

baseline_rate = y_test.mean()
axes[2].axhline(baseline_rate, linestyle='--', color='grey', label=f'No-skill baseline ({baseline_rate:.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
results = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'ROC-AUC': roc_auc_score(y_test, y_proba_lr),
        'PR-AUC':  average_precision_score(y_test, y_proba_lr),
        'F1 (purchase)': f1_score(y_test, y_pred_lr),
    },
    {
        'Model': 'Random Forest',
        'ROC-AUC': roc_auc_score(y_test, y_proba_rf),
        'PR-AUC':  average_precision_score(y_test, y_proba_rf),
        'F1 (purchase)': f1_score(y_test, y_pred_rf),
    },
]).set_index('Model')

print(results.round(4))

## 5. Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['roc_auc', 'average_precision', 'f1']

cv_results = {}
for name, model, X_cv in [
    ('Logistic Regression', lr, X_train_scaled),
    ('Random Forest',        rf, X_train),
]:
    scores = cross_validate(model, X_cv, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {
        'ROC-AUC':       f"{scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}",
        'PR-AUC':        f"{scores['test_average_precision'].mean():.4f} ± {scores['test_average_precision'].std():.4f}",
        'F1 (purchase)': f"{scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}",
    }

cv_df = pd.DataFrame(cv_results).T
print('5-fold stratified CV — mean ± std:')
print(cv_df)

## 6. Permutation Feature Importance

Permutation importance measures how much the model's PR-AUC drops when a feature's values are randomly shuffled.
This is more reliable than MDI (the default `feature_importances_` in scikit-learn), which systematically
overestimates the importance of high-cardinality numerical features.

In [ ]:
result_rf = permutation_importance(
    rf, X_test, y_test,
    n_repeats=10,
    scoring='average_precision',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({
        'feature': FEATURES,
        'importance_mean': result_rf.importances_mean,
        'importance_std':  result_rf.importances_std,
    })
    .sort_values('importance_mean', ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(perm_df['feature'], perm_df['importance_mean'],
        xerr=perm_df['importance_std'], color='#4C72B0', alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean decrease in PR-AUC')
ax.set_title('Permutation feature importance (Random Forest)')
plt.tight_layout()
plt.show()

print(perm_df.sort_values('importance_mean', ascending=False).to_string(index=False))


## 7. Threshold Tuning

The default classification threshold of 0.5 is almost never optimal for imbalanced targets.
We find the threshold that maximises F1 on the test set by sweeping the full precision-recall curve.

Note: in production, threshold tuning should be done on a held-out validation set, not the test set.

In [ ]:
prec, rec, thresholds = precision_recall_curve(y_test, y_proba_rf)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)

best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f'Default threshold (0.50):  F1 = {f1_score(y_test, y_pred_rf):.4f}')
print(f'Optimal threshold ({best_threshold:.3f}): F1 = {best_f1:.4f}')

y_pred_tuned = (y_proba_rf >= best_threshold).astype(int)
print('\nClassification report at optimal threshold:')
print(classification_report(y_test, y_pred_tuned, target_names=['no purchase', 'purchase']))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores[:-1], color='#4C72B0', label='F1')
ax.plot(thresholds, prec[:-1],  color='#55A868', linestyle='--', label='Precision')
ax.plot(thresholds, rec[:-1],   color='#C44E52', linestyle='--', label='Recall')
ax.axvline(best_threshold, color='grey', linestyle=':',
           label=f'Best threshold = {best_threshold:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold vs. F1 / Precision / Recall (Random Forest)')
ax.legend()
plt.tight_layout()
plt.show()


## Summary & Interpretation

| Model | ROC-AUC | PR-AUC | F1 (purchase) |
|-------|---------|--------|---------------|
| Logistic Regression | — | — | — |
| Random Forest | — | — | — |

*(values will fill in after running)*

### Key takeaways
- **Accuracy is misleading** here — a model predicting "no purchase" for every session would achieve 99.19 % accuracy.
- **PR-AUC** and **F1** are the right metrics for imbalanced problems.
- `has_addtocart` and `n_addtocart` are expected to be the strongest predictors — adding items to cart is a clear purchase signal.
- `duration_sec` captures engagement depth that raw event counts miss.

### Possible next steps
- Add time-of-day / day-of-week features from session start time
- Add item-category features (from `item_properties` files)
- Try gradient boosting (XGBoost / LightGBM) for better performance
- Tune classification threshold to optimise Precision vs. Recall trade-off